In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "knowledge_base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from my_rag import search as s

anchors = s._load_anchors()

def check(text, answer_words):
    q = s._encode(text)
    diff = float((s._anchor_vecs @ q).max() - (s._vectors @ q).max())
    results = s.search(text)
    if not results:
        status = "BLOCKED"
        print("        anchor:", anchors[int((s._anchor_vecs @ q).argmax())])
    else:
        status = "NOT FOUND"
        for r in results:
            for w in answer_words:
                if w in r["text"].lower():
                    status = "FOUND"
    sources = []
    for r in results[:3]:
        sources.append(r["source"])
    print(round(diff, 3), "|", status, "|", sources, "|", text)

SUITE = [
    "What is the size of the Suite?",
    "How big is the Suite in square metres?",
    "Does the Suite have a Nile view?",
    "Does the Suite have a Nile view and how big is it?",
    "ما مساحة الجناح؟",
    "الجناح مساحته كام متر؟",
]

TRANSPORT = [
    "What are the transportation options from the hotel to the pyramids and their costs?",
    "How can I get to the pyramids from the hotel?",
    "Does the hotel offer transport to the Giza Pyramids?",
    "How much is a taxi to the pyramids?",
    "Is there a shuttle bus from the hotel to the pyramids?",
    "في مواصلات للأهرامات؟ وبكام؟",
]

OUTSIDE = [
    "Which pyramids should I visit and how much are the tickets?",
    "How much is the entrance ticket to the pyramids?",
    "Recommend a good restaurant in Zamalek",
]

print("RT158 SUITE (answer: 85 m2):")
for t in SUITE:
    check(t, ["85"])
print("\nRT160 TRANSPORT (answer: reception calls a taxi):")
for t in TRANSPORT:
    check(t, ["taxi", "تاكسي"])
print("\nSHOULD BE BLOCKED:")
for t in OUTSIDE:
    check(t, [])

/Users/sondosahmed/projects/HotelAI-Intelligent-Agent_Final_Data/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1126 11:56:25.026000 54452 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 39201.91it/s]


RT158 SUITE (answer: 85 m2):
-0.214 | FOUND | ['room_types.txt', 'room_types.txt', 'room_types.txt'] | What is the size of the Suite?
-0.234 | FOUND | ['room_types.txt', 'room_types.txt', 'room_types.txt'] | How big is the Suite in square metres?
-0.249 | FOUND | ['room_types.txt', 'room_types.txt', 'room_types.txt'] | Does the Suite have a Nile view?
-0.306 | FOUND | ['room_types.txt', 'room_types.txt', 'room_types.txt'] | Does the Suite have a Nile view and how big is it?
-0.109 | FOUND | ['room_types.txt', 'room_amenities.txt', 'room_types.txt'] | ما مساحة الجناح؟
-0.113 | FOUND | ['room_types.txt', 'room_types.txt', 'room_types.txt'] | الجناح مساحته كام متر؟

RT160 TRANSPORT (answer: reception calls a taxi):
0.027 | FOUND | ['location_and_surroundings.txt', 'location_and_surroundings.txt', 'location_and_surroundings.txt'] | What are the transportation options from the hotel to the pyramids and their costs?
-0.049 | FOUND | ['location_and_surroundings.txt', 'location_and_surrounding

In [2]:
import csv
from my_rag.config import FAQ_FILE

EN = "The Giza Pyramids are about 17 km away, 40 to 60 minutes by car. Reception can call a taxi or a ride-hailing car at any hour; the hotel has no fixed price for this trip. The concierge can recommend tours, which are booked with outside operators."

NEW = [
    ("FAQ3930", "What does a trip from the hotel to the pyramids cost?", EN),
    ("FAQ3930", "Is there a fixed price for a taxi from the hotel to the Giza Pyramids?", EN),
]

with open(FAQ_FILE, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    fields = reader.fieldnames
    rows = list(reader)

by_id = {}
questions = set()
max_id = 0
for r in rows:
    by_id[r["faq_id"]] = r
    questions.add(r["question"])
    max_id = max(max_id, int(r["faq_id"].replace("FAQ", "")))

added = []
for base_id, question, answer in NEW:
    if question in questions:
        continue
    max_id += 1
    row = dict(by_id[base_id])
    row["faq_id"] = f"FAQ{max_id}"
    row["question"] = question
    row["answer"] = answer
    row["answer_egyptian"] = ""
    row["fact_id"] = ""
    row["split"] = "train"
    added.append(row)

with open(FAQ_FILE, "rb") as f:
    f.seek(-1, 2)
    last = f.read(1)

with open(FAQ_FILE, "a", encoding="utf-8", newline="") as f:
    if last != b"\n":
        f.write("\r\n")
    writer = csv.DictWriter(f, fieldnames=fields)
    for row in added:
        writer.writerow(row)

print(len(rows), "->", len(rows) + len(added))

4014 -> 4016


In [2]:
from my_rag.evaluate import load_tests

positives, refuse, faq_out = load_tests()

new_outside = [
    "مين هيكسب الدوري السنة دي؟",
    "اعملي ريجيم أخس بيه 5 كيلو",
    "إيه أحسن موبايل تحت 10 آلاف؟",
    "ترجملي الجملة دي للألماني",
    "عايز أعرف نتيجة الثانوية العامة",
    "اكتبلي بوست لينكدإن عن الشغل",
    "وش أفضل مطعم مندي في الرياض؟",
    "كم سعر تذكرة الطيران من جدة للقاهرة؟",
    "What is the capital of Australia?",
    "Can you help me fix my laptop's wifi driver?",
    "Find me a cheap apartment to rent in Maadi",
    "Quel temps fera-t-il à Paris demain ?",
    "Peux-tu m'aider à écrire une lettre de motivation ?",
    "Сколько стоит iPhone в Москве?",
    "Как приготовить борщ?",
]

def measure(label):
    hits = 0
    for item in positives:
        sources = []
        for r in s.search(item["question"]):
            sources.append(r["source"])
        if any(d in sources for d in item["docs"]):
            hits += 1
    old = 0
    for t in refuse + faq_out:
        if not isinstance(t, str):
            t = t["question"]
        if s.search(t) == []:
            old += 1
        else:
            print("    passed:", t)
    new = 0
    for t in new_outside:
        if s.search(t) == []:
            new += 1
    print(label, "| Recall@5:", round(100 * hits / len(positives)), "% | old outside blocked:", old, "of", len(refuse + faq_out), "| new outside blocked:", new, "of", len(new_outside))

full_vecs = s._anchor_vecs
full_anchors = anchors

measure("BEFORE")

keep = []
kept_anchors = []
for i in range(len(full_anchors)):
    text = full_anchors[i].lower()
    if "pyramid" in text or "пирамид" in text:
        print("removed:", full_anchors[i])
        continue
    keep.append(i)
    kept_anchors.append(full_anchors[i])

s._anchor_vecs = full_vecs[keep]
anchors = kept_anchors

measure("AFTER ")

print()
for t in TRANSPORT:
    check(t, ["taxi", "تاكسي"])
for t in OUTSIDE:
    check(t, [])

s._anchor_vecs = full_vecs
anchors = full_anchors

BEFORE | Recall@5: 90 % | old outside blocked: 27 of 27 | new outside blocked: 10 of 15
removed: Which pyramids should I visit and how much are the tickets?
removed: Какие пирамиды стоит посетить и сколько стоят билеты?
    passed: أزور أنهي أهرامات والتذاكر بكام؟
    passed: Quelles pyramides visiter et combien coûtent les billets ?
    passed: تذكرة الأهرامات بكام؟
AFTER  | Recall@5: 90 % | old outside blocked: 24 of 27 | new outside blocked: 10 of 15

-0.115 | FOUND | ['location_and_surroundings.txt', 'location_and_surroundings.txt', 'location_and_surroundings.txt'] | What are the transportation options from the hotel to the pyramids and their costs?
-0.141 | FOUND | ['location_and_surroundings.txt', 'location_and_surroundings.txt', 'location_and_surroundings.txt'] | How can I get to the pyramids from the hotel?
-0.14 | FOUND | ['location_and_surroundings.txt', 'location_and_surroundings.txt', 'location_and_surroundings.txt'] | Does the hotel offer transport to the Giza Pyramids?
-0

In [4]:
import numpy as np
NARROW = [
    "What is the entry fee for the Giza Pyramids site?",
    "Which tourist attractions in Cairo are worth visiting?",
    "Quel est le prix d'entrée du site des pyramides ?",
    "Сколько стоит вход на территорию пирамид?",
]

narrow_vecs = s._model.encode(NARROW, normalize_embeddings=True).astype(full_vecs.dtype)

s._anchor_vecs = np.concatenate([full_vecs[keep], narrow_vecs])
anchors = kept_anchors + NARROW

measure("NARROW")

print()
for t in TRANSPORT:
    check(t, ["taxi", "تاكسي"])
for t in OUTSIDE:
    check(t, [])

s._anchor_vecs = full_vecs
anchors = full_anchors

    passed: أزور أنهي أهرامات والتذاكر بكام؟
NARROW | Recall@5: 90 % | old outside blocked: 26 of 27 | new outside blocked: 10 of 15

        anchor: Quel est le prix d'entrée du site des pyramides ?
0.075 | BLOCKED | [] | What are the transportation options from the hotel to the pyramids and their costs?
0.002 | FOUND | ['location_and_surroundings.txt', 'location_and_surroundings.txt', 'location_and_surroundings.txt'] | How can I get to the pyramids from the hotel?
-0.023 | FOUND | ['location_and_surroundings.txt', 'location_and_surroundings.txt', 'location_and_surroundings.txt'] | Does the hotel offer transport to the Giza Pyramids?
0.056 | FOUND | ['location_and_surroundings.txt', 'location_and_surroundings.txt', 'location_and_surroundings.txt'] | How much is a taxi to the pyramids?
-0.016 | FOUND | ['location_and_surroundings.txt', 'location_and_surroundings.txt', 'location_and_surroundings.txt'] | Is there a shuttle bus from the hotel to the pyramids?
-0.113 | FOUND | ['location_a

In [5]:
NARROW2 = [
    "What is the entry fee for the Giza Pyramids site?",
    "Which tourist attractions in Cairo are worth visiting?",
    "Сколько стоит вход на территорию пирамид?",
]

narrow_vecs = s._model.encode(NARROW2, normalize_embeddings=True).astype(full_vecs.dtype)

s._anchor_vecs = np.concatenate([full_vecs[keep], narrow_vecs])
anchors = kept_anchors + NARROW2

measure("NARROW2")

print()
for t in TRANSPORT:
    check(t, ["taxi", "تاكسي"])
for t in OUTSIDE:
    check(t, [])

s._anchor_vecs = full_vecs
anchors = full_anchors

    passed: أزور أنهي أهرامات والتذاكر بكام؟
NARROW2 | Recall@5: 90 % | old outside blocked: 26 of 27 | new outside blocked: 10 of 15

0.027 | FOUND | ['location_and_surroundings.txt', 'location_and_surroundings.txt', 'location_and_surroundings.txt'] | What are the transportation options from the hotel to the pyramids and their costs?
-0.049 | FOUND | ['location_and_surroundings.txt', 'location_and_surroundings.txt', 'location_and_surroundings.txt'] | How can I get to the pyramids from the hotel?
-0.023 | FOUND | ['location_and_surroundings.txt', 'location_and_surroundings.txt', 'location_and_surroundings.txt'] | Does the hotel offer transport to the Giza Pyramids?
0.033 | FOUND | ['location_and_surroundings.txt', 'location_and_surroundings.txt', 'location_and_surroundings.txt'] | How much is a taxi to the pyramids?
-0.067 | FOUND | ['location_and_surroundings.txt', 'location_and_surroundings.txt', 'location_and_surroundings.txt'] | Is there a shuttle bus from the hotel to the pyramids